# GEO Photometry (star tracking)
Written by Kiyoaki Okudaira<br>
*Kyushu University Hanada Lab / University of Washington / IAU CPS SatHub<br>
(okudaira.kiyoaki.528@s.kyushu-u.ac.jp or kiyoaki@uw.edu)<br>
<br>
Measure brightness of GEO object captured with star tracking method.<br>
<br>
**History**<br>
coding 2026-07-10 : 1st coding<br>
<br>
(c) 2026 Kiyoaki Okudaira - Kyushu University Hanada Lab (SSDL) / University of Washington / IAU CPS SatHub

### Parameters
**Input files settings**

In [ ]:
from pathlib import Path

# Input format: "SER" or "FITS"
input_format = "SER"

# For SER input, specify one SER file. For FITS input, specify a directory.
PATH_input   = Path("/Volumes/SSD-PEU4A/Images/2024-11-22/1977-092K/10_41_00Z.ser")
PATH_output  = Path("/Volumes/SSD-PEU4A/VScode/satphotometry_photometry/output")
PATH_magzero = Path("/Users/kiyoaki/VScode/satphotometry_photometry/output/magzero/magzero_2024-11-22T104100_GAIN3624.json")

**Target and observation information**

In [ ]:
# Artificial-object information
object_name = "EKRAN 2 DEB"
intl_des    = "1977-092K"
norad_id    = "29014"

# Observer and observatory information
observer_name = "Kyushu University Hanada Lab (SSDL)"
observatory   = "KUPT Kyushu University Pegasus Telescope"

**SER files settings**

In [ ]:
# SER frame range (1-based, inclusive). Use None for the first/last frame.
ser_start_frame = 151
ser_end_frame = 910

# Camera settings used when converting SER to FITS.
# Values found in a SharpCap CameraSettings.txt file override these values.
ser_exposure_seconds = 1.0 / 8.0
ser_gain = None
ser_egain = None
ser_egainsav = None
ser_offset = None
ser_x_binning = 1
ser_y_binning = 1
ser_camera_name = None
ser_camera_id = None
ser_ccd_temperature = None
ser_set_temperature = None

**Calibration settings**

In [ ]:
# Calibration files. Set to None when not used.
PATH_dark = None
PATH_flat = None
PATH_flat_dark = None


**Plate solve settings**

In [ ]:
run_astrometry = True
# If False, a failed frame uses the WCS of the nearest successfully solved frame.
require_all_astrometry = False

astrometry_downsample = 2
astrometry_search_radius_deg = 2.0
astrometry_scale_low_arcsec_per_pixel = None
astrometry_scale_high_arcsec_per_pixel = None
astrometry_extra_options = {
    "--objs": 50,
    "--crpix-center": True,
    "--no-remove-lines": True,
    "--uniformize": 0,
    "--overwrite": True,
    "--no-plots": True,
}
astrometry_parallel = True
astrometry_max_workers = 4

**Star subtraction settings**

In [ ]:
# Reference frame used as the common detector grid. None selects the middle frame.
registration_reference_frame = None  # 1-based

# Reprojection interpolation order: "nearest-neighbor", "bilinear", "biquadratic", "bicubic"
registration_order = "bilinear"
registration_parallel = True
registration_max_workers = 4

subtract_stars = True

# Number of approximately evenly spaced registered frames used for the median star image.
# None uses every frame. A larger number improves rejection of the moving GEO object.
median_sample_count = 31

# Save registered frames even when star subtraction is disabled.
save_registered_frames = True

**Measurement settings**

In [ ]:
# Give the target centroid in two REGISTERED frames. Frame numbers are 1-based
# within the selected SER/FITS sequence. Positions use the Astropy/photutils
# convention: the first pixel center is (x, y) = (0, 0).
target_anchor_1_frame = 1
target_anchor_1_x = 3947.0
target_anchor_1_y = 1887.0

target_anchor_2_frame = 760
target_anchor_2_x = 166.0
target_anchor_2_y = 980.0

# Interpolate target position by exposure-midpoint time. If False, use frame index.
target_interpolation_by_time = True

# Optionally refine each interpolated position locally. Normally False because
# the requested method is direct linear interpolation between the two anchors.
refine_interpolated_centroid = False
centroid_half_size = 8

# Aperture photometry settings [pixel]
aperture_radius    = 4
annulus_r_in       = 8
annulus_r_out      = 12

**Software metadata**<br>
Please DO NOT change from original.

In [ ]:
# Output software metadata
editor_app_name  = "SatPhotometry GEO StarTrack"
editor_version = "1.0.0"
creator_app_name = "SatPhotometry GEO StarTrack"
creator_version  = "1.0.0"

### Import and initial settings

In [ ]:
import json
import re
import shutil
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from astropy.stats import SigmaClip, sigma_clipped_stats
from astropy.table import Table
from astropy.time import Time, TimeDelta
from astropy.wcs import WCS
from astropy.wcs.wcs import FITSFixedWarning
from photutils.aperture import (
    ApertureStats,
    CircularAnnulus,
    CircularAperture,
    aperture_photometry,
)
from photutils.centroids import centroid_2dg
from tqdm.auto import tqdm

from satphotometry import astrometry
from satphotometry import serparser

try:
    from reproject import reproject_interp
except ImportError as error:
    raise ImportError(
        "This program requires the 'reproject' package for WCS registration. "
        "Install it with: pip install reproject"
    ) from error

warnings.simplefilter("ignore", FITSFixedWarning)

### General utilities

In [ ]:
def get_fits_files(directory):
    """Return non-hidden FITS files directly contained in a directory."""
    directory = Path(directory)
    if not directory.is_dir():
        raise NotADirectoryError(f"FITS directory does not exist: {directory}")
    files = [
        p for p in directory.iterdir()
        if p.is_file()
        and not p.name.startswith(".")
        and p.suffix.lower() in {".fit", ".fits", ".fts"}
    ]
    files.sort(key=lambda p: p.name.casefold())
    if not files:
        raise FileNotFoundError(f"No FITS files were found in: {directory}")
    return files


def recreate_directory(directory):
    """Delete and recreate a processing directory."""
    directory = Path(directory)
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)
    return directory


def get_observation_time(header):
    """Return exposure start, midpoint, and exposure duration."""
    date_obs = header.get("DATE-OBS", header.get("DATEOBS"))
    if date_obs is None:
        raise KeyError("The FITS header contains neither DATE-OBS nor DATEOBS.")
    if "EXPTIME" not in header:
        raise KeyError("The FITS header does not contain EXPTIME.")

    exptime = float(header["EXPTIME"])
    if not np.isfinite(exptime) or exptime <= 0:
        raise ValueError(f"Invalid EXPTIME: {exptime}")

    try:
        time_start = Time(str(date_obs).strip(), format="fits", scale="utc")
    except ValueError:
        time_start = Time(str(date_obs).strip(), scale="utc")
    time_mid = time_start + TimeDelta(exptime / 2.0, format="sec")
    return time_start, time_mid, exptime


def windows_ticks_to_time(value):
    """Convert SER/Windows ticks since 0001-01-01 to Astropy Time."""
    unix_seconds = float(value) / 10_000_000.0 - 62_135_596_800.0
    return Time(unix_seconds, format="unix", scale="utc")


def parse_number(text):
    """Return the first floating-point number contained in text."""
    if text is None:
        return None
    match = re.search(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", str(text))
    return None if match is None else float(match.group(0))


def parse_sharpcap_settings(path):
    """Read commonly used fields from a SharpCap CameraSettings.txt file."""
    path = Path(path)
    if not path.exists():
        return {}

    settings = {}
    with path.open("r", encoding="utf-8", errors="replace") as file:
        for line in file:
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            settings[key.strip()] = value.strip().strip('"')

    result = {
        "camera_name": settings.get("Camera") or settings.get("CameraName"),
        "camera_id": settings.get("CameraSerialNumber"),
    }

    binning = settings.get("Binning")
    if binning and "x" in binning.lower():
        x_bin, y_bin = re.split("x", binning, maxsplit=1, flags=re.IGNORECASE)
        result["x_binning"] = int(parse_number(x_bin))
        result["y_binning"] = int(parse_number(y_bin))

    exposure_text = settings.get("Exposure")
    exposure_value = parse_number(exposure_text)
    if exposure_value is not None:
        lower = exposure_text.lower()
        if "ms" in lower:
            exposure_value /= 1000.0
        elif "us" in lower or "µs" in lower:
            exposure_value /= 1_000_000.0
        result["exposure"] = exposure_value

    for source_key, destination_key in {
        "Gain": "gain",
        "Offset": "offset",
        "Temperature": "ccd_temperature",
        "Target Temperature": "set_temperature",
    }.items():
        value = parse_number(settings.get(source_key))
        if value is not None:
            result[destination_key] = value

    if "SharpCapVersion" in settings:
        result["swcreate"] = f"SharpCap {settings['SharpCapVersion']}"
    return result

### SER conversion and calibration

In [ ]:
def convert_ser_to_fits(
    ser_path,
    output_directory,
    *,
    start_frame=None,
    end_frame=None,
    exposure_seconds,
    gain=None,
    egain=None,
    egainsav=None,
    offset=None,
    x_binning=1,
    y_binning=1,
    camera_name=None,
    camera_id=None,
    ccd_temperature=None,
    set_temperature=None,
    observer=None,
    observatory=None,
    object_name=None,
    intl_des=None,
    norad_id=None,
):
    """Convert selected SER frames to FITS using satphotometry.serparser."""
    ser_path = Path(ser_path)
    output_directory = recreate_directory(output_directory)
    ser_file = serparser.Serfile(str(ser_path))
    frame_count = int(ser_file.getLength())

    first = 1 if start_frame is None else int(start_frame)
    last = frame_count if end_frame is None else int(end_frame)
    if first < 1 or last > frame_count or first > last:
        raise ValueError(f"Invalid SER frame range: {first}--{last}; total={frame_count}")

    settings = parse_sharpcap_settings(ser_path.with_suffix(".CameraSettings.txt"))
    exposure_seconds = settings.get("exposure", exposure_seconds)
    gain = settings.get("gain", gain)
    offset = settings.get("offset", offset)
    x_binning = settings.get("x_binning", x_binning)
    y_binning = settings.get("y_binning", y_binning)
    camera_name = settings.get("camera_name", camera_name)
    camera_id = settings.get("camera_id", camera_id)
    ccd_temperature = settings.get("ccd_temperature", ccd_temperature)
    set_temperature = settings.get("set_temperature", set_temperature)
    swcreate = settings.get("swcreate", "satphotometry.serparser")

    try:
        timestamps = ser_file.readTrailFromHeader()
        if len(timestamps) != frame_count:
            timestamps = None
    except Exception:
        timestamps = None
    if timestamps is None:
        raise RuntimeError(
            "The SER file does not contain a usable per-frame timestamp trail. "
            "Reliable light-curve timing requires SER timestamps."
        )

    output_files = []
    for zero_index in tqdm(range(first - 1, last), desc="Converting SER to FITS"):
        output_path = output_directory / f"{ser_path.stem}_{zero_index + 1:05d}.fits"
        ser_file.setCurrentPosition(zero_index)
        ser_file.saveFit(str(output_path))

        with fits.open(output_path, mode="update", memmap=False) as hdul:
            header = hdul[0].header
            header["DATE-OBS"] = (
                windows_ticks_to_time(timestamps[zero_index]).fits,
                "Exposure start time UTC",
            )
            header["EXPTIME"] = (float(exposure_seconds), "Exposure time [s]")
            header["XBINNING"] = int(x_binning)
            header["YBINNING"] = int(y_binning)
            header["SWCREATE"] = swcreate
            optional = {
                "GAIN": gain,
                "EGAIN": egain,
                "EGAINSAV": egainsav,
                "OFFSET": offset,
                "INSTRUME": camera_name,
                "CAMID": camera_id,
                "CCD-TEMP": ccd_temperature,
                "SET-TEMP": set_temperature,
                "OBSERVER": observer,
                "OBSERVAT": observatory,
                "OBJECT": object_name,
                "INTL-DES": intl_des,
                "NORAD-ID": norad_id,
            }
            for key, value in optional.items():
                if value is not None:
                    header[key] = value
            hdul.flush(output_verify="silentfix")
        output_files.append(output_path)
    return output_files


def create_master_dark(path):
    if path is None:
        return None
    path = Path(path)
    files = get_fits_files(path) if path.is_dir() else [path]
    stack = []
    for file_path in files:
        with fits.open(file_path, memmap=False) as hdul:
            stack.append(np.asarray(hdul[0].data, dtype=float))
    return np.nanmedian(np.stack(stack), axis=0)


def create_master_flat(path, master_flat_dark=None):
    if path is None:
        return None
    path = Path(path)
    files = get_fits_files(path) if path.is_dir() else [path]
    stack = []
    for file_path in files:
        with fits.open(file_path, memmap=False) as hdul:
            data = np.asarray(hdul[0].data, dtype=float)
        if master_flat_dark is not None:
            data = data - master_flat_dark
        stack.append(data)
    master = np.nanmedian(np.stack(stack), axis=0)
    norm = np.nanmedian(master[np.isfinite(master)])
    if not np.isfinite(norm) or norm == 0:
        raise ValueError("The master flat cannot be normalized.")
    return master / norm


def calibrate_fits_files(fits_files, dark=None, flat=None):
    if dark is None and flat is None:
        return
    for file_path in tqdm(fits_files, desc="Calibrating FITS frames"):
        with fits.open(file_path, mode="update", memmap=False) as hdul:
            data = np.asarray(hdul[0].data, dtype=float)
            if dark is not None:
                data = data - dark
            if flat is not None:
                with np.errstate(divide="ignore", invalid="ignore"):
                    data = data / flat
            hdul[0].data = data
            hdul.flush(output_verify="silentfix")

### Plate solving

In [ ]:
def build_astrometry_options(output_path, work_directory, previous_radec=None):
    """Build solve-field options for satphotometry.astrometry.platesolve."""
    work_directory = Path(work_directory)
    work_directory.mkdir(parents=True, exist_ok=True)
    options = {
        "-N": str(output_path),
        "--dir": str(work_directory),
        "--downsample": int(astrometry_downsample),
    }
    options.update(astrometry_extra_options)

    if astrometry_scale_low_arcsec_per_pixel is not None:
        options["--scale-low"] = astrometry_scale_low_arcsec_per_pixel
        options["--scale-units"] = "arcsecperpix"
    if astrometry_scale_high_arcsec_per_pixel is not None:
        options["--scale-high"] = astrometry_scale_high_arcsec_per_pixel
        options["--scale-units"] = "arcsecperpix"
    if previous_radec is not None:
        options["--ra"] = float(previous_radec[0])
        options["--dec"] = float(previous_radec[1])
        options["--radius"] = float(astrometry_search_radius_deg)
    return options


def solve_one_frame(task):
    frame_index, file_path, solved_path, frame_work, reference_radec = task
    file_path = Path(file_path)
    solved_path = Path(solved_path)
    try:
        options = build_astrometry_options(solved_path, frame_work, reference_radec)
        result_path, solved = astrometry.platesolve(
            str(file_path), options, async_process=False, jupyter_env=True
        )
        result_path = Path(result_path)
        if not solved or not result_path.exists():
            return frame_index, file_path, result_path, False, "solve-field did not solve the frame."
        return frame_index, file_path, result_path, True, None
    except Exception as error:
        return frame_index, file_path, solved_path, False, str(error)


def solve_reference_frame(file_path, solved_directory, work_directory):
    file_path = Path(file_path)
    solved_path = Path(solved_directory) / file_path.name
    options = build_astrometry_options(
        solved_path,
        Path(work_directory) / f"{file_path.stem}_reference",
        previous_radec=None,
    )
    result_path, solved = astrometry.platesolve(
        str(file_path), options, async_process=True, jupyter_env=True
    )
    if not solved or not Path(result_path).exists():
        raise RuntimeError("The astrometry reference frame could not be plate-solved.")

    with fits.open(result_path, memmap=False) as hdul:
        shape = hdul[0].data.shape
        header = hdul[0].header.copy()
    wcs = WCS(header)
    sky = wcs.pixel_to_world((shape[1] - 1) / 2.0, (shape[0] - 1) / 2.0)
    return Path(result_path), (float(sky.ra.deg), float(sky.dec.deg))


def plate_solve_files(fits_files, solved_directory, work_directory, parallel=True, max_workers=4):
    """Plate-solve all frames using one common RA/Dec search center."""
    solved_directory = recreate_directory(solved_directory)
    work_directory = recreate_directory(work_directory)
    fits_files = [Path(path) for path in fits_files]
    if not fits_files:
        raise ValueError("No FITS files were supplied.")

    reference_index = len(fits_files) // 2
    _, reference_radec = solve_reference_frame(
        fits_files[reference_index], solved_directory, work_directory
    )
    print(
        f"Reference astrometry center: RA={reference_radec[0]:.6f} deg, "
        f"Dec={reference_radec[1]:.6f} deg"
    )

    tasks = [
        (
            i,
            str(path),
            str(solved_directory / path.name),
            str(work_directory / path.stem),
            reference_radec,
        )
        for i, path in enumerate(fits_files)
    ]

    if parallel and int(max_workers) > 1:
        raw_results = []
        with ThreadPoolExecutor(max_workers=int(max_workers)) as executor:
            futures = {executor.submit(solve_one_frame, task): task[0] for task in tasks}
            for future in tqdm(as_completed(futures), total=len(futures), desc="Plate solving"):
                raw_results.append(future.result())
        raw_results.sort(key=lambda item: item[0])
    else:
        raw_results = [solve_one_frame(task) for task in tqdm(tasks, desc="Plate solving")]

    records = []
    failures = []
    for frame_index, source_path, solved_path, solved, error in raw_results:
        header = None
        wcs = None
        if solved:
            try:
                with fits.open(solved_path, memmap=False) as hdul:
                    header = hdul[0].header.copy()
                wcs = WCS(header)
                if not wcs.has_celestial:
                    raise ValueError("Solved header has no celestial WCS.")
            except Exception as read_error:
                solved = False
                error = str(read_error)
                wcs = None
        if not solved:
            failures.append((frame_index, Path(source_path).name, error))
            print(f"Plate solve failed for {Path(source_path).name}: {error}")
        records.append({
            "frame_index": frame_index,
            "source_path": Path(source_path),
            "solved_path": Path(solved_path),
            "solved": bool(solved),
            "wcs": wcs,
            "header": header,
        })

    if require_all_astrometry and failures:
        names = ", ".join(item[1] for item in failures[:10])
        suffix = " ..." if len(failures) > 10 else ""
        raise RuntimeError(
            f"Plate solve failed for {len(failures)} frame(s): {names}{suffix}. "
            "WCS registration requires every frame to be solved."
        )

    if failures:
        records = complement_failed_wcs(records)

    return records


def complement_failed_wcs(records):
    """
    Give each failed frame the WCS of the nearest successfully solved frame.

    The failed frame keeps its own image data, timestamp, exposure, and camera
    metadata. Only the celestial WCS is copied. The complemented FITS file is
    written to the frame's normal solved-path location so that the downstream
    registration code can process every frame in the same way.
    """
    successful_indices = [
        index for index, record in enumerate(records)
        if record["solved"] and record["wcs"] is not None
    ]
    if not successful_indices:
        raise RuntimeError(
            "No frame was plate-solved successfully, so failed-frame WCS "
            "complementation is impossible."
        )

    complemented_records = []
    for index, record in enumerate(records):
        if record["solved"] and record["wcs"] is not None:
            complemented_records.append({
                **record,
                "plate_solved": True,
                "wcs_complemented": False,
                "wcs_reference_index": index,
            })
            continue

        nearest_index = min(
            successful_indices,
            key=lambda solved_index: (abs(solved_index - index), solved_index),
        )
        nearest_record = records[nearest_index]

        with fits.open(record["source_path"], memmap=False) as hdul:
            source_data = np.asarray(hdul[0].data)
            source_header = hdul[0].header.copy()

        nearest_header = nearest_record["header"].copy()
        nearest_wcs_header = WCS(nearest_header).to_header(relax=True)
        wcs_keys = set(nearest_wcs_header.keys())

        # Start from the nearest successful frame so that all WCS-related
        # cards, including distortion information retained in that header,
        # remain available. Then restore non-WCS metadata from this frame.
        complemented_header = nearest_header.copy()
        for card in source_header.cards:
            key = card.keyword
            if key in {"", "COMMENT", "HISTORY"} or key in wcs_keys:
                continue
            try:
                complemented_header[key] = (card.value, card.comment)
            except Exception:
                pass

        # Always restore frame-specific timing metadata from the original frame.
        for time_key in (
            "DATE-OBS",
            "DATEOBS",
            "MJD-OBS",
            "JD",
            "TIMESYS",
            "TIMEUNIT",
            "EXPTIME",
        ):
            if time_key in source_header:
                complemented_header[time_key] = (
                    source_header[time_key],
                    source_header.comments[time_key],
                )

        complemented_header["WCSCOMP"] = (
            True,
            "WCS copied from nearest solved frame",
        )
        complemented_header["WCSREF"] = (
            int(nearest_index),
            "Zero-based WCS source frame index",
        )
        complemented_header["WCSOFF"] = (
            int(nearest_index - index),
            "WCS source index minus current index",
        )

        solved_path = Path(record["solved_path"])
        solved_path.parent.mkdir(parents=True, exist_ok=True)
        fits.writeto(
            solved_path,
            source_data,
            complemented_header,
            overwrite=True,
            output_verify="silentfix",
        )

        complemented_wcs = WCS(complemented_header)
        print(
            f"Using WCS from nearest solved frame {nearest_index + 1} for "
            f"failed frame {index + 1}: {record['source_path'].name}"
        )
        complemented_records.append({
            **record,
            "solved_path": solved_path,
            "solved": True,
            "plate_solved": False,
            "wcs": complemented_wcs,
            "header": complemented_header,
            "wcs_complemented": True,
            "wcs_reference_index": nearest_index,
        })

    return complemented_records

### Star subtraction

In [ ]:
def select_registration_reference(records):
    """Return the configured valid registration-reference index."""
    if registration_reference_frame is None:
        index = len(records) // 2
    else:
        index = int(registration_reference_frame) - 1
    if index < 0 or index >= len(records):
        raise IndexError("registration_reference_frame is outside the selected frame range.")
    if not records[index]["solved"]:
        raise RuntimeError("The selected registration reference frame was not plate-solved.")
    return index


def register_one_frame(task):
    """Reproject one solved image to the common reference WCS/grid."""
    record, reference_header, reference_shape, reference_index, output_directory = task
    output_path = Path(output_directory) / record["source_path"].name
    try:
        with fits.open(record["solved_path"], memmap=False) as hdul:
            source_data = np.asarray(hdul[0].data, dtype=float)
            source_header = hdul[0].header.copy()

        registered, footprint = reproject_interp(
            (source_data, WCS(source_header)),
            WCS(reference_header),
            shape_out=reference_shape,
            order=registration_order,
            return_footprint=True,
        )
        registered[np.asarray(footprint) <= 0] = np.nan

        output_header = reference_header.copy()

        # Keep non-WCS metadata from the current source frame.
        wcs_keys = set(
            WCS(reference_header)
            .to_header(relax=True)
            .keys()
        )

        for card in source_header.cards:
            key = card.keyword

            if key in {"", "COMMENT", "HISTORY"} or key in wcs_keys:
                continue

            try:
                output_header[key] = (
                    card.value,
                    card.comment,
                )
            except Exception:
                pass

        # Always restore the source frame's timing metadata.
        for time_key in (
            "DATE-OBS",
            "DATEOBS",
            "MJD-OBS",
            "JD",
            "TIMESYS",
            "TIMEUNIT",
            "EXPTIME",
        ):
            if time_key in source_header:
                output_header[time_key] = (
                    source_header[time_key],
                    source_header.comments[time_key],
                )

        # Keep timing/camera metadata from the source frame while retaining the reference WCS.
        wcs_keys = set(WCS(reference_header).to_header(relax=True).keys())
        for card in source_header.cards:
            key = card.keyword
            if key in {"", "COMMENT", "HISTORY"} or key in wcs_keys:
                continue
            try:
                output_header[key] = (card.value, card.comment)
            except Exception:
                pass
        output_header["WCSREG"] = (True, "Reprojected to common reference WCS")
        output_header["REGREF"] = (
            int(reference_index),
            "Zero-based registration reference index",
        )
        fits.writeto(
            output_path,
            registered,
            output_header,
            overwrite=True,
            output_verify="silentfix",
        )
        return {**record, "registered_path": output_path, "registration_error": None}
    except Exception as error:
        return {**record, "registered_path": None, "registration_error": str(error)}


def register_frames(records, output_directory, parallel=True, max_workers=4):
    """Register every frame onto the WCS and pixel grid of one reference frame."""
    output_directory = recreate_directory(output_directory)
    reference_index = select_registration_reference(records)
    reference_record = records[reference_index]
    with fits.open(reference_record["solved_path"], memmap=False) as hdul:
        reference_shape = hdul[0].data.shape
        reference_header = hdul[0].header.copy()

    tasks = [
        (record, reference_header, reference_shape, reference_index, output_directory)
        for record in records
    ]
    output_records = [None] * len(records)

    if parallel and int(max_workers) > 1:
        with ThreadPoolExecutor(max_workers=int(max_workers)) as executor:
            futures = {
                executor.submit(register_one_frame, task): task[0]["frame_index"]
                for task in tasks
            }
            for future in tqdm(
                as_completed(futures), total=len(futures), desc="Registering frames"
            ):
                index = futures[future]
                output_records[index] = future.result()
    else:
        for task in tqdm(tasks, desc="Registering frames"):
            result = register_one_frame(task)
            output_records[result["frame_index"]] = result

    errors = [record for record in output_records if record["registration_error"]]
    if errors:
        first = errors[0]
        raise RuntimeError(
            f"WCS registration failed for {len(errors)} frame(s). First failure: "
            f"{first['source_path'].name}: {first['registration_error']}"
        )
    return output_records, reference_index, reference_header


def choose_median_sample_indices(frame_count, sample_count):
    if frame_count < 3:
        raise ValueError("At least three frames are required to create a median star image.")
    if sample_count is None or int(sample_count) >= frame_count:
        return np.arange(frame_count, dtype=int)
    count = int(sample_count)
    if count < 3:
        raise ValueError("median_sample_count must be None or at least 3.")
    return np.unique(np.linspace(0, frame_count - 1, count).round().astype(int))


def create_median_star_image(records, output_path, sample_count=None):
    """Create a median field-star image from registered frames."""
    indices = choose_median_sample_indices(len(records), sample_count)
    stack = []
    for index in tqdm(indices, desc="Loading median samples"):
        with fits.open(records[index]["registered_path"], memmap=False) as hdul:
            stack.append(np.asarray(hdul[0].data, dtype=float))
    median_data = np.nanmedian(np.stack(stack, axis=0), axis=0)

    with fits.open(records[indices[0]]["registered_path"], memmap=False) as hdul:
        header = hdul[0].header.copy()
    header["MEDSTAR"] = (True, "Median field-star reference image")
    header["MEDNSAMP"] = (len(indices), "Number of median input frames")
    fits.writeto(output_path, median_data, header, overwrite=True, output_verify="silentfix")
    return median_data, indices


def subtract_median_one_frame(task):
    record, median_data, output_directory = task
    output_path = Path(output_directory) / record["source_path"].name
    with fits.open(record["registered_path"], memmap=False) as hdul:
        data = np.asarray(hdul[0].data, dtype=float)
        header = hdul[0].header.copy()
    result = data - median_data
    header["STARSUB"] = (True, "Registered median field-star image subtracted")
    fits.writeto(output_path, result, header, overwrite=True, output_verify="silentfix")
    return {**record, "photometry_path": output_path}


def subtract_median_star_image(records, median_data, output_directory, parallel=True, max_workers=4):
    """Subtract the same registered median star image from every frame."""
    output_directory = recreate_directory(output_directory)
    tasks = [(record, median_data, output_directory) for record in records]
    output_records = [None] * len(records)
    if parallel and int(max_workers) > 1:
        with ThreadPoolExecutor(max_workers=int(max_workers)) as executor:
            futures = {
                executor.submit(subtract_median_one_frame, task): task[0]["frame_index"]
                for task in tasks
            }
            for future in tqdm(
                as_completed(futures), total=len(futures), desc="Subtracting median stars"
            ):
                index = futures[future]
                output_records[index] = future.result()
    else:
        for task in tqdm(tasks, desc="Subtracting median stars"):
            result = subtract_median_one_frame(task)
            output_records[result["frame_index"]] = result
    return output_records

### Photometry

In [ ]:
def read_frame_mid_times(records):
    """Read midpoint times from the registered image headers."""
    times = []
    for record in records:
        with fits.open(record["registered_path"], memmap=False) as hdul:
            _, time_mid, _ = get_observation_time(hdul[0].header)
        times.append(time_mid)
    return times


def interpolate_target_positions(records, registration_header):
    """
    Estimate target positions from two anchors measured on
    fits_raw or fits_solved.

    First interpolate the target position on the original detector
    coordinate system. Then transform each estimated position to the
    common registered WCS.
    """
    count = len(records)

    # Anchor frame numbers are 1-based within the selected frame range.
    i1 = int(target_anchor_1_frame) - 1
    i2 = int(target_anchor_2_frame) - 1

    if i1 < 0 or i1 >= count or i2 < 0 or i2 >= count:
        raise IndexError(
            "A target anchor frame is outside the selected frame range."
        )

    if i1 == i2:
        raise ValueError(
            "The two target anchor frames must be different."
        )

    x1 = float(target_anchor_1_x)
    y1 = float(target_anchor_1_y)
    x2 = float(target_anchor_2_x)
    y2 = float(target_anchor_2_y)

    if not np.all(np.isfinite([x1, y1, x2, y2])):
        raise ValueError(
            "All target anchor coordinates must be finite numbers."
        )

    # --------------------------------------------------------
    # Build interpolation coordinate
    # --------------------------------------------------------

    if target_interpolation_by_time:
        mid_times = read_frame_mid_times(records)

        origin = mid_times[0]

        coordinate = np.array(
            [
                (time - origin).to_value("sec")
                for time in mid_times
            ],
            dtype=float,
        )

        method = "source_linear_time_to_registered_wcs"

    else:
        coordinate = np.arange(
            count,
            dtype=float,
        )

        method = "source_linear_frame_to_registered_wcs"

    denominator = (
        coordinate[i2]
        - coordinate[i1]
    )

    # Fall back to frame-number interpolation when timestamps
    # of the two anchor frames are identical.
    if (
        not np.isfinite(denominator)
        or np.isclose(denominator, 0.0)
    ):
        if target_interpolation_by_time:
            print(
                "Warning: the two target-anchor frames have "
                "identical midpoint times. Falling back to "
                "frame-number interpolation."
            )

            coordinate = np.arange(
                count,
                dtype=float,
            )

            method = (
                "source_linear_frame_fallback_to_registered_wcs"
            )

            denominator = (
                coordinate[i2]
                - coordinate[i1]
            )

        if (
            not np.isfinite(denominator)
            or np.isclose(denominator, 0.0)
        ):
            raise ValueError(
                "The two target anchors have identical "
                "interpolation coordinates."
            )

    # --------------------------------------------------------
    # Interpolate positions on fits_raw / fits_solved
    # --------------------------------------------------------

    fraction = (
        coordinate
        - coordinate[i1]
    ) / denominator

    source_x = (
        x1
        + fraction * (x2 - x1)
    )

    source_y = (
        y1
        + fraction * (y2 - y1)
    )

    # --------------------------------------------------------
    # Transform each source-frame position to registered WCS
    # --------------------------------------------------------

    reference_wcs = WCS(
        registration_header
    )

    registered_x = np.full(
        count,
        np.nan,
        dtype=float,
    )

    registered_y = np.full(
        count,
        np.nan,
        dtype=float,
    )

    for index, record in enumerate(records):
        # solved_path contains either:
        # - the frame's own WCS, or
        # - the nearest successful frame's WCS when solve failed.
        with fits.open(
            record["solved_path"],
            memmap=False,
        ) as hdul:
            source_header = hdul[0].header.copy()

        source_wcs = WCS(
            source_header
        )

        # Original detector position -> celestial coordinate
        sky_position = source_wcs.pixel_to_world(
            float(source_x[index]),
            float(source_y[index]),
        )

        # Celestial coordinate -> registered detector position
        x_registered, y_registered = (
            reference_wcs.world_to_pixel(
                sky_position
            )
        )

        registered_x[index] = float(
            x_registered
        )

        registered_y[index] = float(
            y_registered
        )

    if (
        not np.all(np.isfinite(registered_x))
        or not np.all(np.isfinite(registered_y))
    ):
        raise RuntimeError(
            "At least one target position could not be "
            "transformed to the registered WCS."
        )

    print(
        "Target anchors interpreted on fits_raw/fits_solved "
        "and transformed to the registered WCS."
    )

    print(
        f"  Anchor 1 registered position: "
        f"x={registered_x[i1]:.3f}, "
        f"y={registered_y[i1]:.3f}"
    )

    print(
        f"  Anchor 2 registered position: "
        f"x={registered_x[i2]:.3f}, "
        f"y={registered_y[i2]:.3f}"
    )

    return (
        registered_x,
        registered_y,
        source_x,
        source_y,
        method,
    )


def refine_centroid(image, x, y, half_size=8):
    image = np.asarray(image, dtype=float)
    ny, nx = image.shape
    xc, yc = int(round(float(x))), int(round(float(y)))
    x0, x1 = max(0, xc - half_size), min(nx, xc + half_size + 1)
    y0, y1 = max(0, yc - half_size), min(ny, yc + half_size + 1)
    cutout = image[y0:y1, x0:x1]
    finite = np.isfinite(cutout)
    if np.count_nonzero(finite) < 9:
        raise ValueError("Too few finite pixels for centroid refinement.")
    _, median, _ = sigma_clipped_stats(cutout[finite], sigma=3.0, maxiters=5)
    source = np.where(finite, np.clip(cutout - median, 0, None), 0.0)
    x_local, y_local = centroid_2dg(source)
    return float(x0 + x_local), float(y0 + y_local)


def measure_object(image, x_object, y_object, aperture_radius, annulus_r_in, annulus_r_out):
    """Perform circular-aperture photometry with a sigma-clipped annulus."""
    image = np.asarray(image, dtype=float)
    ny, nx = image.shape
    margin = float(annulus_r_out)
    if not (margin <= x_object < nx - margin and margin <= y_object < ny - margin):
        raise ValueError(
            f"Target/annulus lies outside the registered frame: x={x_object:.3f}, y={y_object:.3f}"
        )

    position = (float(x_object), float(y_object))
    aperture = CircularAperture(position, r=float(aperture_radius))
    annulus = CircularAnnulus(position, r_in=float(annulus_r_in), r_out=float(annulus_r_out))
    phot_table = aperture_photometry(image, aperture, method="exact")
    aperture_sum = float(phot_table["aperture_sum"][0])
    annulus_stats = ApertureStats(
        image, annulus, sigma_clip=SigmaClip(sigma=3.0, maxiters=5)
    )
    background_median = float(annulus_stats.median)
    aperture_area = float(aperture.area)
    background_sum = background_median * aperture_area
    object_flux = aperture_sum - background_sum
    return {
        "aperture_sum": aperture_sum,
        "background_median": background_median,
        "background_sum": background_sum,
        "aperture_area": aperture_area,
        "object_flux": object_flux,
    }


def read_photometric_zeropoint(magzero_path):
    """Prefer ZP_elec; otherwise use ZP_exp without gain correction."""
    with Path(magzero_path).open("r", encoding="utf-8") as file:
        data = json.load(file)
    source = data.get("phot_params", data)

    zp_elec = source.get("photometric_zeropoint_elec", source.get("ZP_elec"))
    try:
        zp_elec = float(zp_elec)
    except (TypeError, ValueError):
        zp_elec = np.nan
    if np.isfinite(zp_elec):
        return data, zp_elec, "elec"

    zp_exp = source.get("photometric_zeropoint_exp", source.get("ZP_exp"))
    try:
        zp_exp = float(zp_exp)
    except (TypeError, ValueError):
        zp_exp = np.nan
    if np.isfinite(zp_exp):
        return data, zp_exp, "exp"

    raise KeyError(
        "Neither photometric_zeropoint_elec nor photometric_zeropoint_exp is available."
    )


def calculate_apparent_magnitude(object_flux, exptime, header, zeropoint, mode):
    if not np.isfinite(object_flux) or object_flux <= 0:
        return np.nan, np.nan, np.nan
    instrumental_mag = -2.5 * np.log10(object_flux)
    if mode == "elec":
        if "EGAINSAV" not in header:
            raise KeyError(
                "photometric_zeropoint_elec is selected, but EGAINSAV is missing "
                "from the science FITS header."
            )
        egainsav = float(header["EGAINSAV"])
        if not np.isfinite(egainsav) or egainsav <= 0:
            raise ValueError(f"Invalid EGAINSAV: {egainsav}")
        apparent_mag = (
            instrumental_mag
            + 2.5 * np.log10(exptime)
            - 2.5 * np.log10(egainsav)
            + zeropoint
        )
    elif mode == "exp":
        egainsav = np.nan
        apparent_mag = instrumental_mag + 2.5 * np.log10(exptime) + zeropoint
    else:
        raise ValueError(f"Unknown zeropoint mode: {mode}")
    return instrumental_mag, apparent_mag, egainsav

### Output

In [ ]:
def json_value(value):
    if np.ma.is_masked(value):
        return None
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    return value


def export_kupt_lightcurve_json(table, output_path, fits_header, magzero_data, zeropoint_mode):
    """Write a KUPT-lightcurveJSON version 2 product."""
    valid_rows = [row for row in table if str(row["time_start_utc"]).strip()]
    if not valid_rows:
        raise ValueError("No valid timestamp is available for JSON output.")

    first, last = valid_rows[0], valid_rows[-1]
    start = Time(str(first["time_start_utc"]), scale="utc")
    end = Time(str(last["time_start_utc"]), scale="utc") + TimeDelta(
        float(last["exptime_seconds"]), format="sec"
    )
    phot_source = magzero_data.get("phot_params", magzero_data)

    data_rows = []
    for row in table:
        data_rows.append({
            "id": int(row["frame_index"]),
            "time_start_utc": str(row["time_start_utc"]) or None,
            "time_mid_utc": str(row["time_mid_utc"]) or None,
            "elapsed_seconds": json_value(row["elapsed_seconds"]),
            "x_centroid": json_value(row["x_centroid"]),
            "y_centroid": json_value(row["y_centroid"]),
            "aperture_sum": json_value(row["aperture_sum_adu"]),
            "bkg_med_pix": json_value(row["background_median_adu_per_pixel"]),
            "object_flux": json_value(row["object_flux_adu"]),
            "mag_in": json_value(row["instrumental_mag"]),
            "mag_app": json_value(row["apparent_mag"]),
            "status": str(row["status"]),
            "comment": {"general": [
                f"Source FITS file: {row['filename']}",
                f"Position method: {row['position_method']}",
                f"Zeropoint mode: {zeropoint_mode}",
            ]},
        })

    output = {
        "header": {
            "object": {
                "objectName": object_name,
                "intlDES": intl_des,
                "noradID": None if norad_id is None else str(norad_id),
            },
            "startDateTime": start.isot,
            "endDateTime": end.isot,
            "length": len(table),
            "captureSettings": {
                "Xnaxis": json_value(fits_header.get("NAXIS1")),
                "Ynaxis": json_value(fits_header.get("NAXIS2")),
                "Xbinning": json_value(fits_header.get("XBINNING", 1)),
                "Ybinning": json_value(fits_header.get("YBINNING", 1)),
                "exposure": json_value(fits_header.get("EXPTIME")),
                "gain": json_value(fits_header.get("GAIN")),
                "egain": json_value(fits_header.get("EGAIN")),
                "egainsav": json_value(fits_header.get("EGAINSAV")),
                "offset": json_value(fits_header.get("OFFSET")),
                "camName": json_value(fits_header.get("INSTRUME")),
                "camID": json_value(fits_header.get("CAMID")),
                "swcreate": json_value(fits_header.get("SWCREATE")),
            },
            "photParams": {
                "pixel_area_arcsec2": json_value(phot_source.get("pixel_area_arcsec2")),
                "aperture_area_arcsec2": json_value(phot_source.get("aperture_area_arcsec2")),
                "ZP": json_value(phot_source.get("photometric_zeropoint")),
                "ZP_exp": json_value(phot_source.get("photometric_zeropoint_exp")),
                "ZP_elec": json_value(phot_source.get("photometric_zeropoint_elec")),
                "ZP_scatter": json_value(phot_source.get("zeropoint_scatter")),
                "selected_mode": zeropoint_mode,
            },
            "observer": {"observerName": observer_name, "observatory": observatory},
            "editor": {"appName": editor_app_name, "version": editor_version},
            "measureSettings": {
                "measureFieldShape": "CIRCLE",
                "measureFieldSize": {
                    "aperture_radius_pixel": float(aperture_radius),
                    "annulus_r_in_pixel": float(annulus_r_in),
                    "annulus_r_out_pixel": float(annulus_r_out),
                },
                "targetPosition": {
                    "method": "two-anchor linear interpolation on registered frames",
                    "anchor1": {
                        "frame": int(target_anchor_1_frame),
                        "x": float(target_anchor_1_x),
                        "y": float(target_anchor_1_y),
                    },
                    "anchor2": {
                        "frame": int(target_anchor_2_frame),
                        "x": float(target_anchor_2_x),
                        "y": float(target_anchor_2_y),
                    },
                    "interpolateByTime": bool(target_interpolation_by_time),
                },
            },
        },
        "data": data_rows,
        "createdBy": {
            "appName": creator_app_name,
            "version": creator_version,
            "built": int(datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")),
        },
        "dataType": "KUPT-lightcurveJSON",
        "version": 2,
        "comment": {
            "general": [
                "GEO sidereal-tracking photometry with per-frame plate solving, "
                "WCS registration, and registered median field-star subtraction."
            ],
            "TLE": [],
        },
    }

    for key in ("captureSettings", "photParams"):
        output["header"][key] = {
            k: v for k, v in output["header"][key].items() if v is not None
        }

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(output, file, ensure_ascii=False, indent=2, allow_nan=False)
    return output

### Main pipeline

In [ ]:
def create_geo_star_tracking_lightcurve():
    """Run the complete GEO sidereal-tracking photometry pipeline."""
    if not run_astrometry:
        raise ValueError("run_astrometry must be True because WCS registration is required.")

    PATH_output.mkdir(parents=True, exist_ok=True)
    raw_directory = PATH_output / "fits_raw"
    solved_directory = PATH_output / "fits_solved"
    astrometry_work_directory = PATH_output / "astrometry_work"
    registered_directory = PATH_output / "fits_registered"
    star_subtracted_directory = PATH_output / "fits_star_subtracted"
    product_directory = PATH_output / "lightcurve"
    product_directory.mkdir(parents=True, exist_ok=True)

    if input_format.upper() == "SER":
        fits_files = convert_ser_to_fits(
            PATH_input,
            raw_directory,
            start_frame=ser_start_frame,
            end_frame=ser_end_frame,
            exposure_seconds=ser_exposure_seconds,
            gain=ser_gain,
            egain=ser_egain,
            egainsav=ser_egainsav,
            offset=ser_offset,
            x_binning=ser_x_binning,
            y_binning=ser_y_binning,
            camera_name=ser_camera_name,
            camera_id=ser_camera_id,
            ccd_temperature=ser_ccd_temperature,
            set_temperature=ser_set_temperature,
            observer=observer_name,
            observatory=observatory,
            object_name=object_name,
            intl_des=intl_des,
            norad_id=norad_id,
        )
    elif input_format.upper() == "FITS":
        source_files = get_fits_files(PATH_input)
        recreate_directory(raw_directory)
        fits_files = []
        for source in source_files:
            destination = raw_directory / source.name
            shutil.copy2(source, destination)
            fits_files.append(destination)
    else:
        raise ValueError("input_format must be either 'SER' or 'FITS'.")

    master_flat_dark = create_master_dark(PATH_flat_dark)
    master_dark = create_master_dark(PATH_dark)
    master_flat = create_master_flat(PATH_flat, master_flat_dark)
    calibrate_fits_files(fits_files, dark=master_dark, flat=master_flat)

    astrometry_records = plate_solve_files(
        fits_files,
        solved_directory,
        astrometry_work_directory,
        parallel=astrometry_parallel,
        max_workers=astrometry_max_workers,
    )

    registered_records, registration_reference_index, registration_header = register_frames(
        astrometry_records,
        registered_directory,
        parallel=registration_parallel,
        max_workers=registration_max_workers,
    )
    print(f"Registration reference frame: {registration_reference_index + 1}")

    median_path = PATH_output / "median_field_stars.fits"
    if subtract_stars:
        median_data, median_indices = create_median_star_image(
            registered_records, median_path, sample_count=median_sample_count
        )
        print(
            f"Median star image uses {len(median_indices)} frame(s): "
            f"{median_indices[0] + 1} ... {median_indices[-1] + 1}"
        )
        photometry_records = subtract_median_star_image(
            registered_records,
            median_data,
            star_subtracted_directory,
            parallel=registration_parallel,
            max_workers=registration_max_workers,
        )
    else:
        photometry_records = [
            {**record, "photometry_path": record["registered_path"]}
            for record in registered_records
        ]

    if not save_registered_frames and registered_directory.exists():
        shutil.rmtree(registered_directory)

    (
        target_x_values,
        target_y_values,
        target_source_x_values,
        target_source_y_values,
        position_method,
    ) = interpolate_target_positions(
        registered_records,
        registration_header,
    )

    # Save a registered sum image to make manual anchor checking easier.
    sum_data = None
    for record in tqdm(photometry_records, desc="Creating registered sum"):
        with fits.open(record["photometry_path"], memmap=False) as hdul:
            data = np.asarray(hdul[0].data, dtype=float)
        sum_data = np.nan_to_num(data, nan=0.0) if sum_data is None else sum_data + np.nan_to_num(data, nan=0.0)
    fits.writeto(
        PATH_output / "registered_star_subtracted_sum.fits",
        sum_data,
        registration_header,
        overwrite=True,
        output_verify="silentfix",
    )

    magzero_data, zeropoint, zeropoint_mode = read_photometric_zeropoint(PATH_magzero)
    print(f"Using zeropoint mode: {zeropoint_mode}")

    results = []
    first_time_mid = None
    first_header = None
    for frame_index, record in enumerate(tqdm(photometry_records, desc="Measuring GEO target")):
        path = record["photometry_path"]
        status = "Success"
        x_object = float(target_x_values[frame_index])
        y_object = float(target_y_values[frame_index])
        actual_position_method = position_method
        try:
            with fits.open(path, memmap=False) as hdul:
                image = np.asarray(hdul[0].data, dtype=float)
                header = hdul[0].header.copy()
            if first_header is None:
                first_header = header.copy()

            time_start, time_mid, exptime = get_observation_time(header)
            if first_time_mid is None:
                first_time_mid = time_mid
            elapsed = float((time_mid - first_time_mid).to_value("sec"))

            if refine_interpolated_centroid:
                x_object, y_object = refine_centroid(
                    image, x_object, y_object, half_size=centroid_half_size
                )
                actual_position_method += "+local_centroid"

            phot = measure_object(
                image,
                x_object,
                y_object,
                aperture_radius,
                annulus_r_in,
                annulus_r_out,
            )
            instrumental_mag, apparent_mag, egainsav = calculate_apparent_magnitude(
                phot["object_flux"], exptime, header, zeropoint, zeropoint_mode
            )
            if not np.isfinite(apparent_mag):
                status = "Non-positive flux"

            results.append({
                "frame_index": frame_index,
                "filename": path.name,
                "time_start_utc": time_start.isot,
                "time_mid_utc": time_mid.isot,
                "elapsed_seconds": elapsed,
                "exptime_seconds": exptime,
                "egainsav_electron_per_adu": egainsav,
                "x_centroid": x_object,
                "y_centroid": y_object,
                "position_method": actual_position_method,
                "registration_reference_index": registration_reference_index,
                "aperture_radius_pixel": aperture_radius,
                "annulus_r_in_pixel": annulus_r_in,
                "annulus_r_out_pixel": annulus_r_out,
                "aperture_area_pixel2": phot["aperture_area"],
                "aperture_sum_adu": phot["aperture_sum"],
                "background_median_adu_per_pixel": phot["background_median"],
                "background_sum_adu": phot["background_sum"],
                "object_flux_adu": phot["object_flux"],
                "instrumental_mag": instrumental_mag,
                "apparent_mag": apparent_mag,
                "zeropoint_mode": zeropoint_mode,
                "status": status,
                "source_x_estimate": float(
                    target_source_x_values[frame_index]
                ),

                "source_y_estimate": float(
                    target_source_y_values[frame_index]
                ),

                "x_centroid": x_object,
                "y_centroid": y_object,

                "position_method": actual_position_method,
            })
        except Exception as error:
            print(f"Frame {frame_index + 1} ({path.name}) failed: {error}")
            results.append({
                "frame_index": frame_index,
                "filename": path.name,
                "time_start_utc": "",
                "time_mid_utc": "",
                "elapsed_seconds": np.nan,
                "exptime_seconds": np.nan,
                "egainsav_electron_per_adu": np.nan,
                "x_centroid": x_object,
                "y_centroid": y_object,
                "position_method": actual_position_method,
                "registration_reference_index": registration_reference_index,
                "aperture_radius_pixel": aperture_radius,
                "annulus_r_in_pixel": annulus_r_in,
                "annulus_r_out_pixel": annulus_r_out,
                "aperture_area_pixel2": np.nan,
                "aperture_sum_adu": np.nan,
                "background_median_adu_per_pixel": np.nan,
                "background_sum_adu": np.nan,
                "object_flux_adu": np.nan,
                "instrumental_mag": np.nan,
                "apparent_mag": np.nan,
                "zeropoint_mode": zeropoint_mode,
                "status": f"Error: {error}",
            })

    table = Table(rows=results)
    valid = [row for row in results if str(row["time_start_utc"]).strip()]
    if not valid:
        raise RuntimeError("No frame was measured successfully.")

    start = Time(valid[0]["time_start_utc"], scale="utc")
    end = Time(valid[-1]["time_start_utc"], scale="utc") + TimeDelta(
        valid[-1]["exptime_seconds"], format="sec"
    )
    safe_object = re.sub(r"[^A-Za-z0-9._-]+", "_", object_name).strip("_")
    safe_intldes = re.sub(r"[^A-Za-z0-9._-]+", "_", intl_des).strip("_")
    base_name = (
        f"lightcurve_{safe_object}_{safe_intldes}_"
        f"{start.datetime.strftime('%Y-%m-%d_%H%M%S')}_"
        f"{end.datetime.strftime('%H%M%S')}"
    )
    csv_path = product_directory / f"{base_name}.csv"
    json_path = product_directory / f"{base_name}.json"
    table.write(csv_path, format="ascii.csv", overwrite=True)
    json_output = export_kupt_lightcurve_json(
        table, json_path, first_header, magzero_data, zeropoint_mode
    )
    print(f"Saved CSV:  {csv_path}")
    print(f"Saved JSON: {json_path}")
    return table, csv_path, json_path, json_output

### Run

In [ ]:
lightcurve_table, output_csv_path, output_json_path, kupt_json_data = (
    create_geo_star_tracking_lightcurve()
)

### Result

In [ ]:
elapsed_seconds = np.asarray(lightcurve_table["elapsed_seconds"], dtype=float)
apparent_mag = np.asarray(lightcurve_table["apparent_mag"], dtype=float)
valid = np.isfinite(elapsed_seconds) & np.isfinite(apparent_mag)

plt.figure(figsize=(9, 5))
plt.plot(
    elapsed_seconds[valid],
    apparent_mag[valid],
    marker="o",
    markersize=4,
    linewidth=1,
)
plt.xlabel("Elapsed time from first frame [s]")
plt.ylabel("Apparent magnitude [mag]")
plt.gca().invert_yaxis()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()